In [ ]:
'''
Title: Data Wrangling Project
Name: Sandra Lung'ahi
Date: 23rd September, 2025

Description: A project to practice data wrangling concepts using the Netflix dataset.
'''

In [ ]:
# Import libraries
import pandas as pd
import numpy as np


In [ ]:
# Load Dataset
df = pd.read_csv("/kaggle/input/netflix-shows/netflix_titles.csv")
print("Dataset loaded successfully")

In [ ]:
# Data Discovery

# Print dataset information summary.
print("\nDataset Info\n")
df.info()

# Print the shape of the dataset.
print("\nShape (Rows x Columns):", df.shape)

# Print the list of all column names
print("\nColumns:", df.columns.tolist())

# Print the data types of each column.
print("\nData types:\n", df.dtypes)

# Print the number of missing values per column.
print("\nMissing values per column:\n", df.isnull().sum())

# Print the total number of duplicate rows in the dataset.
print("\nNumber of duplicate rows:", df.duplicated().sum())

# Print the first 5 rows.
print("\nSample rows:\n", df.head())


In [ ]:
# Structuring

# Convert 'date_added' to datetime
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

# Extract duration value & unit.
df[['duration_value', 'duration_unit']] = df['duration'].str.extract(r'(\d+)\s*(\w+)')
df['duration_value'] = pd.to_numeric(df['duration_value'], errors='coerce')

print("\nStructured Columns Sample\n", df[['duration', 'duration_value', 'duration_unit']].head())

In [ ]:
# Data Cleaning and Imputation

# Remove duplicates
df = df.drop_duplicates()

# Create director-cast pair column
df['dir_cast'] = df['director'].fillna('') + '---' + df['cast'].fillna('')

# Count frequency of director-cast pairs
pair_counts = df['dir_cast'].value_counts()
valid_pairs = pair_counts[pair_counts >= 3].index

# Build mapping for cast → director.
dir_cast_map = {}
for pair in valid_pairs:
    director, cast = pair.split('---')
    if cast not in dir_cast_map:   # assign first valid mapping
        dir_cast_map[cast] = director

# Impute missing directors using cast
df['director'] = df.apply(
    lambda row: dir_cast_map.get(row['cast'], row['director']), axis=1
)

# Fill remaining missing directors
df['director'] = df['director'].fillna('Not Given')

# Imputation of Country.
dir_country_map = (
    df.dropna(subset=['director', 'country'])
      .drop_duplicates(subset=['director'])
      .set_index('director')['country']
      .to_dict()
)

df['country'] = df.apply(
    lambda row: dir_country_map.get(row['director'], row['country']), axis=1
)

df['country'] = df['country'].fillna('Unknown')

# Cast
df['cast'] = df['cast'].fillna('Not Given')

# Rating
df['rating'] = df['rating'].fillna('Not Rated')

# Date Added 
df['date_added'] = df['date_added'].fillna(df['date_added'].mode()[0])

# Drop helper column
df.drop(columns=['dir_cast'], inplace=True)

# Normalize season labels
df['duration_unit'] = df['duration_unit'].replace({'Seasons': 'Season'})

# Fill missing duration with 'Unknown'
df['duration'] = df['duration'].fillna('Unknown')

# For missing duration_value, also fill defaults
df['duration_value'] = df['duration_value'].fillna(0).astype(int)
df['duration_unit'] = df['duration_unit'].fillna('Unknown')

print("\nMissing values after imputation\n", df.isnull().sum())

# Print the first 5 rows.
print("\nSample rows:\n", df.head())

In [ ]:
# Error Checks

import datetime as dt

# Find rows where date_added < release_year
invalid_years = df['date_added'].dt.year < df['release_year']
print("Number of inconsistent rows:", invalid_years.sum())

# Inspect some inconsistent rows
print("\nInconsistent Sample\n", 
      df.loc[invalid_years, ['title', 'date_added', 'release_year']].head())

# Fix by replacing release_year with date_added year
df.loc[invalid_years, 'release_year'] = df.loc[invalid_years, 'date_added'].dt.year

# Confirm fix
print("\nRemaining inconsistencies:", 
      (df['date_added'].dt.year < df['release_year']).sum())


In [ ]:
# Validation

# Drop any helper columns used for wrangling
df = df.drop(columns=['dir_cast'], errors='ignore')

# Check column data types
print("\nData Types\n", df.dtypes)

# Ensure date_added is datetime
assert np.issubdtype(df['date_added'].dtype, np.datetime64)

# Ensure duration_value is numeric
assert pd.api.types.is_numeric_dtype(df['duration_value'])

# remove rows where release_year < 1997
df = df[df['release_year'] >= 1997]

# Check for missing fields
print("\nMissing Values Check\n", df.isnull().sum())

# Sample rows for visual inspection
print("\nSample Records\n", df.sample(5))

# reset index
df_reset = df.reset_index(drop=True)


In [ ]:
# Publish

# Save cleaned dataset
df_reset.to_csv('/kaggle/working/cleaned_netflix.csv', index=False)
print("Cleaned dataset saved as cleaned_netflix.csv")
